In [ ]:
import soundfile as sf
import numpy as np
import pandas as pd
import glob
import os
import time
import matplotlib
matplotlib.use("Agg")          # non-GUI backend, tidak perlu display
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm import tqdm
from datetime import datetime, timedelta

sampled_root = '../data/sampled'
sampled2_root = '../data/sampled2'

In [ ]:
df = pd.read_csv('dataHasilConvert.csv')
df

,flac_path,wav_path,format_flac,format_wav,subtype_flac,subtype_wav,sr,duration_s,size_flac_kb,size_wav_kb
0,../data/sampled\0\00\0002114d3f.flac,../data/sampled\0\00\0002114d3f.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.200,67.57,131.29
1,../data/sampled\0\00\00079b49f9.flac,../data/sampled\0\00\00079b49f9.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.400,71.20,168.79
2,../data/sampled\0\00\00084ff1da.flac,../data/sampled\0\00\00084ff1da.wav,FLAC,WAV,PCM_16,PCM_16,16000,7.100,91.39,221.92
3,../data/sampled\0\00\000923633f.flac,../data/sampled\0\00\000923633f.wav,FLAC,WAV,PCM_16,PCM_16,16000,4.100,84.85,128.17
4,../data/sampled\0\00\0009d31e8f.flac,../data/sampled\0\00\0009d31e8f.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.300,96.10,165.67
...,...,...,...,...,...,...,...,...,...,...
18384,../data/sampled\9\9f\9fe3d15862.flac,../data/sampled\9\9f\9fe3d15862.wav,FLAC,WAV,PCM_16,PCM_16,16000,8.800,136.22,275.04
18385,../data/sampled\9\9f\9ff3410471.flac,../data/sampled\9\9f\9ff3410471.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.758,107.56,180.00
18386,../data/sampled\9\9f\9ff6f34230.flac,../data/sampled\9\9f\9ff6f34230.wav,FLAC,WAV,PCM_16,PCM_16,16000,5.500,98.99,171.92
18387,../data/sampled\9\9f\9ff789f862.flac,../data/sampled\9\9f\9ff789f862.wav,FLAC,WAV,PCM_16,PCM_16,16000,3.700,62.33,115.67


In [ ]:
def sampling(wav_path: str) -> tuple[np.ndarray, int]:
    samples, sr = sf.read(wav_path, dtype="float64")

    if samples.ndim == 2:       # stereo jadi mono 
        samples = samples[:, 0]

    return samples, sr

In [ ]:
def quantization(samples: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Step 1 – level rentang -32768 sampai +32767
    max_level = 32_767
    min_level = -32_768

    # Step 2 – kode PCM: amplitudo kontinu × 32767 kemudian bulatkan
    kode_pcm = np.clip(np.round(samples * max_level), min_level, max_level).astype(int)

    # Step 3 – rekonstruksi kode PCM ÷ 32767 = amplitudo diskrit
    quantized = kode_pcm / max_level

    return kode_pcm, quantized 

In [ ]:
# all_paths = [
#     os.path.abspath(os.path.join(root_dir, file))
#     for root_dir, dirs, files in os.walk(sampled_root)
#     for file in files
# ]
# print(all_paths)

In [ ]:
# hasil_quantization = []
# hasil_sampling =[]
# for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
    
#     samples, sr = sampling(wav_path)

#     kode_pcm, quantized = quantization(samples)
#     wave = os.path.splitext(os.path.basename(wav_path))[0]
#     hasil_sampling.append({
#                 "Id"   : wave,
#                 "Sample Rate" : f" {sr} Hz",
#                 "Jumlah Sampel" : f"{len(samples)}",
#                 "Durasi"     : f"{len(samples)/sr:.1f} detik",
#                 "Contoh Sampling" : (samples[:5])
#             })

#     hasil_quantization.append({
#         "Id"      : wave,
#         "Sample Rate" : f" {sr} Hz",
#         "Jumlah Sampel" : f"{len(samples)}",
#         "Durasi"     : f"{len(samples)/sr:.1f} detik",
#         "Contoh Sampling"   : samples[:5].tolist(),     # amplitudo kontinu
#         "Contoh kode PCM"   : kode_pcm[:5].tolist(),   # hasil step 2
#         "Contoh Quantized"  : quantized[:5].tolist(),  # hasil step 3
#     })
# df_sampling = pd.DataFrame(hasil_sampling)
# df_sampling.to_csv("HasilSampling.csv", index=False)
# df_quantization = pd.DataFrame(hasil_quantization)
# df_quantization.to_csv("HasilQuantization.csv", index=False)

In [ ]:
# df_sampling

In [ ]:
# df_quantization

In [ ]:
def voice_activity_detection(
    quantized  : np.ndarray,
    sr         : int,
    frame_ms   : int = 300,     # durasi frame: 300 ms menghasilkan 480 sampel (dokumen)
    snr_min    : float = 6.0,   # ambang batas bawah  
    snr_max    : float = 10.0,  # ambang batas atas   
) -> tuple[np.ndarray, list[dict]]:

    #Bagi sinyal menjadi frame 
    frame_size = int(sr * frame_ms / 1000)      # 16000 * 300/1000 = 480
    frames     = [
        quantized[i : i + frame_size]
        for i in range(0, len(quantized), frame_size)
        if len(quantized[i : i + frame_size]) == frame_size  # buang frame tidak lengkap
    ]

    frame_results = []
    voice_frames  = []

    for idx, frame in enumerate(frames):

        # Step 1: Hitung Energi per Frame 
        # Dokumen: "Setiap nilai amplitudo diskrit dipangkatkan dan ditotalkan"
        energi = np.sum(frame ** 2)

        # Step 2: Perhitungan Daya 
        N    = len(frame)
        daya = energi / N

        # Step 3: Estimasi Daya 
        delta         = 1 / 32_767
        estimasi_daya = (delta ** 2) / 12

        # Step 4: Rasio Signal to Noise
        snr = daya / estimasi_daya if estimasi_daya > 0 else 0

        # Step 5: Konversi SNR ke Logaritmik (dB) 
        snr_db = 10 * np.log10(snr) if snr > 0 else 0

        # Step 6: Perbandingan terhadap Ambang Batas 
        if snr_db > snr_max:
            label = "wicara"
            voice_frames.append(frame)
        elif snr_db < snr_min:
            label = "noise"
        else:
            label = "tidak pasti"

        frame_results.append({
            "frame_idx"    : idx,
            "energi"       : round(energi, 10),
            "daya"         : round(daya, 10),
            "estimasi_daya": round(estimasi_daya, 10),
            "snr"          : round(snr, 4),
            "snr_db"       : round(snr_db, 10),
            "label"        : label,
        })
    # Gabungkan semua frame wicara menjadi satu sinyal wicara utuh
    sinyal_bersih = np.concatenate(voice_frames) if voice_frames else np.array([])

    return sinyal_bersih, frame_results

In [ ]:
# hasil_sampling     = []
# hasil_quantization = []
# hasil_vad          = []
# hasil_vad_detail   = []

# for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
#     try:
#         audio_id = os.path.splitext(os.path.basename(wav_path))[0]

#         # SAMPLING 
#         samples, sr = sampling(wav_path)

#         hasil_sampling.append({
#             "Id"              : audio_id,
#             "Sample Rate"     : f"{sr} Hz",
#             "Jumlah Sampel"   : len(samples),
#             "Durasi"          : f"{len(samples)/sr:.1f} detik",
#             "Contoh Sampling" : samples[:5].tolist(),
#         })

#         # QUANTIZATION 
#         kode_pcm, quantized = quantization(samples)

#         hasil_quantization.append({
#             "Id"              : audio_id,
#             "Sample Rate"     : f"{sr} Hz",
#             "Jumlah Sampel"   : len(samples),
#             "Durasi"          : f"{len(samples)/sr:.1f} detik",
#             "Contoh Sampling" : samples[:5].tolist(),
#             "Contoh Kode PCM" : kode_pcm[:5].tolist(),
#             "Contoh Quantized": quantized[:5].tolist(),
#         })

#         # VOICE ACTIVITY DETECTION
#         sinyal_bersih, frame_results = voice_activity_detection(quantized, sr)

#         df_frame = pd.DataFrame(frame_results)
#         hasil_vad.append({
#             "Id"             : audio_id,
#             "Total Frame"    : len(frame_results),
#             "Frame Wicara"   : df_frame[df_frame["label"] == "wicara"].shape[0],
#             "Frame Noise"    : df_frame[df_frame["label"] == "noise"].shape[0],
#             "Frame Tdk Pasti": df_frame[df_frame["label"] == "tidak pasti"].shape[0],
#             "Sinyal Bersih"  : sinyal_bersih[:5].tolist() if len(sinyal_bersih) > 0 else [],
#         })
#         for frame in frame_results:
#             hasil_vad_detail.append({
#                 "Id"           : audio_id,   
#                 "frame_idx"    : frame["frame_idx"],
#                 "energi"       : frame["energi"],
#                 "daya"         : frame["daya"],
#                 "estimasi_daya": frame["estimasi_daya"],
#                 "snr"          : frame["snr"],
#                 "snr_db"       : frame["snr_db"],
#                 "label"        : frame["label"],
#             })
#     except Exception as e:
#         print(f"Error pada file : {wav_path} : {e}")

# df_sampling     = pd.DataFrame(hasil_sampling)
# df_quantization = pd.DataFrame(hasil_quantization)
# df_vad          = pd.DataFrame(hasil_vad)
# df_vad_detail = pd.DataFrame(hasil_vad_detail)   

# df_sampling.to_csv("HasilSampling.csv", index=False)
# df_quantization.to_csv("HasilQuantization.csv", index=False)
# df_vad.to_csv("HasilVAD.csv", index=False)
# df_vad_detail.to_csv("HasilVAD_detail.csv", index=False)


# print(f"HasilSampling.csv     = {len(df_sampling)} audio")
# print(f"HasilQuantization.csv = {len(df_quantization)} audio")
# print(f"HasilVAD.csv          = {len(df_vad)} audio")
# print(f"HasilVAD_detail.csv   = {len(df_vad)} audio")

Error pada file : ../data/sampled\5\59\5980ab3cf8.wav : 'label'
HasilSampling.csv     = 18389 audio
HasilQuantization.csv = 18389 audio
HasilVAD.csv          = 18388 audio
HasilVAD_detail.csv   = 18388 audio


In [ ]:
hasil_sampling     = []
hasil_quantization = []
hasil_vad          = []
hasil_vad_detail   = []

for wav_path in glob.glob(os.path.join(sampled2_root, "**/*.wav"), recursive=True):
    try:
        audio_id = os.path.splitext(os.path.basename(wav_path))[0]

        # SAMPLING 
        samples, sr = sampling(wav_path)

        hasil_sampling.append({
            "Id"              : audio_id,
            "Sample Rate"     : f"{sr} Hz",
            "Jumlah Sampel"   : len(samples),
            "Durasi"          : f"{len(samples)/sr:.1f} detik",
            "Contoh Sampling" : samples[:5].tolist(),
        })

        # QUANTIZATION 
        kode_pcm, quantized = quantization(samples)

        hasil_quantization.append({
            "Id"              : audio_id,
            "Sample Rate"     : f"{sr} Hz",
            "Jumlah Sampel"   : len(samples),
            "Durasi"          : f"{len(samples)/sr:.1f} detik",
            "Contoh Sampling" : samples[:5].tolist(),
            "Contoh Kode PCM" : kode_pcm[:5].tolist(),
            "Contoh Quantized": quantized[:5].tolist(),
        })

        # VOICE ACTIVITY DETECTION
        sinyal_bersih, frame_results = voice_activity_detection(quantized, sr)

        df_frame = pd.DataFrame(frame_results)
        hasil_vad.append({
            "Id"             : audio_id,
            "Total Frame"    : len(frame_results),
            "Frame Wicara"   : df_frame[df_frame["label"] == "wicara"].shape[0],
            "Frame Noise"    : df_frame[df_frame["label"] == "noise"].shape[0],
            "Frame Tdk Pasti": df_frame[df_frame["label"] == "tidak pasti"].shape[0],
            "Sinyal Bersih"  : sinyal_bersih[:5].tolist() if len(sinyal_bersih) > 0 else [],
        })
        for frame in frame_results:
            hasil_vad_detail.append({
                "Id"           : audio_id,   
                "frame_idx"    : frame["frame_idx"],
                "energi"       : frame["energi"],
                "daya"         : frame["daya"],
                "estimasi_daya": frame["estimasi_daya"],
                "snr"          : frame["snr"],
                "snr_db"       : frame["snr_db"],
                "label"        : frame["label"],
            })
    except Exception as e:
        print(f"Error pada file : {wav_path} : {e}")

df_sampling     = pd.DataFrame(hasil_sampling)
df_quantization = pd.DataFrame(hasil_quantization)
df_vad          = pd.DataFrame(hasil_vad)
df_vad_detail = pd.DataFrame(hasil_vad_detail)   

df_sampling.to_csv("HasilSampling2.csv", index=False)
df_quantization.to_csv("HasilQuantization2.csv", index=False)
df_vad.to_csv("HasilVAD2.csv", index=False)
df_vad_detail.to_csv("HasilVAD_detail2.csv", index=False)


print(f"HasilSampling2.csv     = {len(df_sampling)} audio")
print(f"HasilQuantization2.csv = {len(df_quantization)} audio")
print(f"HasilVA2.csv          = {len(df_vad)} audio")
print(f"HasilVAD_detail2.csv   = {len(df_vad)} audio")

In [ ]:

viz_root = "visualizations2"
os.makedirs(viz_root, exist_ok=True)

# ── KONSTANTA VISUALISASI ─────────────────────────────────────────────────────
MAX_POINTS   = 2000    # maksimum titik yang diplot (downsample jika lebih)
N_MARK       = 30      # jumlah titik sampel yang ditandai
DPI          = 80      # resolusi output (turunkan untuk lebih ringan)
FIG_W        = 8       # lebar figure
FIG_H        = 2.5     # tinggi figure

# ── HELPER ────────────────────────────────────────────────────────────────────

def get_hex_folder(audio_id):
    hex_id    = format(hash(audio_id) & 0xFFFF, '04x')
    return hex_id[0], hex_id[0:2]


def downsample(arr, max_pts):
    """Ambil max_pts titik merata dari array — tanpa komputasi berat."""
    if len(arr) <= max_pts:
        return np.arange(len(arr)), arr
    idx = np.linspace(0, len(arr) - 1, max_pts, dtype=int)
    return idx, arr[idx]


def save_visualization(audio_id, samples, sr, kode_pcm, quantized, sinyal_bersih, frame_results):
    folder, subfolder = get_hex_folder(audio_id)

    for subdir in ["waveform", "sampling", "quantization", "vad"]:
        os.makedirs(os.path.join(viz_root, folder, subfolder, subdir), exist_ok=True)

    N      = len(samples)
    dur    = N / sr

    # Downsample sekali, pakai ulang di semua plot
    vis_idx, vis_s = downsample(samples, MAX_POINTS)
    vis_t          = vis_idx / sr
    vis_q          = quantized[vis_idx]

    # Titik sampel yang ditandai (subset kecil dari vis)
    mark_pos  = np.linspace(0, len(vis_idx) - 1, N_MARK, dtype=int)
    mark_t    = vis_t[mark_pos]
    mark_s    = vis_s[mark_pos]
    mark_q    = vis_q[mark_pos]
    step_t    = vis_t[1] - vis_t[0] if len(vis_t) > 1 else 1.0

    # ── 1. WAVEFORM ───────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot(vis_t, vis_s, color="#185FA5", linewidth=0.5, rasterized=True)
    ax.fill_between(vis_t, vis_s, alpha=0.15, color="#185FA5", rasterized=True)
    ax.axhline(0, color="gray", linewidth=0.4, linestyle="--")
    ax.set_title(f"Waveform — {audio_id}", fontsize=9)
    ax.set_xlabel("Waktu (s)", fontsize=8)
    ax.set_ylabel("Amplitudo", fontsize=8)
    ax.set_xlim(0, dur)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.15, linewidth=0.4)
    fig.tight_layout(pad=0.5)
    fig.savefig(os.path.join(viz_root, folder, subfolder, "waveform",
                             f"{audio_id}.png"), dpi=DPI)
    plt.close(fig)

    # ── 2. SAMPLING ───────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot(vis_t, vis_s, color="#185FA5", linewidth=0.5,
            alpha=0.3, rasterized=True, label="Sinyal")
    ax.vlines(mark_t, 0, mark_s,
              colors="#BA7517", linewidth=0.6, alpha=0.5, linestyles="dashed")
    ax.plot(mark_t, mark_s, "o",
            color="#BA7517", markersize=3, zorder=5, label="Sampel")
    ax.axhline(0, color="gray", linewidth=0.4, linestyle="--")
    ax.set_title(f"Sampling — {audio_id}", fontsize=9)
    ax.set_xlabel("Waktu (s)", fontsize=8)
    ax.set_ylabel("Amplitudo", fontsize=8)
    ax.set_xlim(0, dur)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.15, linewidth=0.4)
    fig.tight_layout(pad=0.5)
    fig.savefig(os.path.join(viz_root, folder, subfolder, "sampling",
                             f"{audio_id}.png"), dpi=DPI)
    plt.close(fig)

    # ── 3. QUANTIZATION ───────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    ax.plot(vis_t, vis_s, color="#185FA5", linewidth=0.5,
            alpha=0.2, rasterized=True, label="Asli")
    ax.step(mark_t, mark_q, where="mid",
            color="#0F6E56", linewidth=1.2, label="Quantized", zorder=4)
    ax.plot(mark_t, mark_q, "o",
            color="#0F6E56", markersize=2.5, zorder=5)
    # Error shading — hanya N_MARK kotak kecil, ringan
    half = step_t * 0.5
    for i in range(len(mark_pos)):
        if mark_s[i] != mark_q[i]:
            ax.fill_between(
                [mark_t[i] - half, mark_t[i] + half],
                mark_s[i], mark_q[i],
                color="#C32D2D", alpha=0.15
            )
    ax.axhline(0, color="gray", linewidth=0.4, linestyle="--")
    ax.set_title(f"Quantization — {audio_id}", fontsize=9)
    ax.set_xlabel("Waktu (s)", fontsize=8)
    ax.set_ylabel("Amplitudo", fontsize=8)
    ax.set_xlim(0, dur)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.15, linewidth=0.4)
    fig.tight_layout(pad=0.5)
    fig.savefig(os.path.join(viz_root, folder, subfolder, "quantization",
                             f"{audio_id}.png"), dpi=DPI)
    plt.close(fig)

    # ── 4. VAD ────────────────────────────────────────────────────────────────
    if frame_results:
        df_f      = pd.DataFrame(frame_results)
        labels    = df_f["label"].values
        snr_vals  = df_f["snr_db"].values
        color_map = {"wicara": "#185FA5", "noise": "#C32D2D", "tidak pasti": "#BA7517"}
        c_list    = [color_map.get(l, "gray") for l in labels]

        fig, axes = plt.subplots(2, 1, figsize=(FIG_W, 4),
                                 gridspec_kw={"height_ratios": [2, 1]})

        # Panel atas — sinyal bersih (downsample juga)
        if len(sinyal_bersih) > 0:
            _, sb_vis = downsample(sinyal_bersih, MAX_POINTS)
            sb_t      = np.linspace(0, dur, len(sb_vis))
            axes[0].plot(sb_t, sb_vis, color="#185FA5",
                         linewidth=0.5, alpha=0.8, rasterized=True)
            axes[0].fill_between(sb_t, sb_vis, alpha=0.12,
                                 color="#185FA5", rasterized=True)
        axes[0].set_title(f"VAD — Sinyal Bersih — {audio_id}", fontsize=9)
        axes[0].set_ylabel("Amplitudo", fontsize=8)
        axes[0].tick_params(labelsize=7)
        axes[0].grid(True, alpha=0.15, linewidth=0.4)

        # Panel bawah — bar SNR per frame
        axes[1].bar(range(len(labels)), snr_vals,
                    color=c_list, width=1.0, edgecolor="none", alpha=0.8)
        axes[1].set_ylabel("SNR (dB)", fontsize=8)
        axes[1].set_xlabel("Frame", fontsize=8)
        axes[1].tick_params(labelsize=7)
        axes[1].grid(True, alpha=0.15, linewidth=0.4)
        axes[1].legend(
            handles=[Patch(color=v, label=k) for k, v in color_map.items()],
            fontsize=7, loc="upper right"
        )

        fig.tight_layout(pad=0.5)
        fig.savefig(os.path.join(viz_root, folder, subfolder, "vad",
                                 f"{audio_id}.png"), dpi=DPI)
        plt.close(fig)


# ── BATCH + ISTIRAHAT ─────────────────────────────────────────────────────────

def istirahat(menit, batch_ke, total_batch):
    selesai_pada = datetime.now() + timedelta(minutes=menit)
    print(f"\n  {'─'*45}")
    print(f"  Batch {batch_ke}/{total_batch} selesai. Istirahat {menit} menit...")
    print(f"  Lanjut pada : {selesai_pada.strftime('%H:%M:%S')}")
    print(f"  {'─'*45}")
    for sisa in range(menit * 60, 0, -1):
        mnt, dtk = divmod(sisa, 60)
        print(f"\r  Sisa waktu  : {mnt:02d}:{dtk:02d}", end="", flush=True)
        time.sleep(1)
    print(f"\r  Istirahat selesai. Melanjutkan batch {batch_ke + 1}...{' '*10}\n")


# ── MAIN ──────────────────────────────────────────────────────────────────────

JUMLAH_BATCH  = 30
ISTIRAHAT_MIN = 5

wav_files = glob.glob(os.path.join(sampled2_root, "**/*.wav"), recursive=True)
total     = len(wav_files)
errors    = []

if total == 0:
    print("Tidak ada file WAV ditemukan.")
else:
    ukuran_batch = max(1, -(-total // JUMLAH_BATCH))
    batches      = [wav_files[i:i + ukuran_batch] for i in range(0, total, ukuran_batch)]
    total_batch  = len(batches)

    print(f"{'='*47}")
    print(f"  Total file WAV      : {total}")
    print(f"  Jumlah batch        : {total_batch}")
    print(f"  Ukuran per batch    : ~{ukuran_batch} file")
    print(f"  Istirahat per batch : {ISTIRAHAT_MIN} menit")
    print(f"  Total istirahat     : {(total_batch - 1) * ISTIRAHAT_MIN} menit")
    print(f"{'='*47}\n")

    mulai_total = datetime.now()

    for b_idx, batch in enumerate(batches, start=1):
        print(f"\n  ▶ Batch {b_idx}/{total_batch}  ({len(batch)} file)")
        print(f"    Mulai : {datetime.now().strftime('%H:%M:%S')}\n")

        for wav_path in tqdm(batch, desc=f"  Batch {b_idx:02d}", unit="file", ncols=70):
            audio_id = os.path.splitext(os.path.basename(wav_path))[0]
            try:
                samples, sr                  = sampling(wav_path)
                kode_pcm, quantized          = quantization(samples)
                sinyal_bersih, frame_results = voice_activity_detection(quantized, sr)

                save_visualization(
                    audio_id, samples, sr,
                    kode_pcm, quantized,
                    sinyal_bersih, frame_results
                )

            except Exception as e:
                errors.append({
                    "audio_id" : audio_id,
                    "file"     : wav_path,
                    "error"    : str(e),
                    "waktu"    : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })
                tqdm.write(f"  [SKIP] {audio_id} — {e}")
                continue

        if b_idx < total_batch:
            istirahat(ISTIRAHAT_MIN, b_idx, total_batch)

    # ── RINGKASAN ─────────────────────────────────────────────────────────────
    durasi_total = datetime.now() - mulai_total
    total_viz    = total - len(errors)

    print(f"\n{'='*47}")
    print(f"  SELESAI — {str(durasi_total).split('.')[0]}")
    print(f"{'─'*47}")
    print(f"  Visualisasi tersimpan : {total_viz:>5} audio")
    print(f"  Dilewati (error)      : {len(errors):>5} file")
    print(f"  Output folder         : {viz_root}/")
    print(f"{'='*47}")

    if errors:
        df_errors = pd.DataFrame(errors)
        df_errors.to_csv("ErrorVisualisasi2.csv", index=False)
        print(f"\n  Error log → ErrorVisualisasi2.csv")
        print(df_errors[["audio_id", "error"]].to_string(index=False))

  Total file WAV      : 18389
  Jumlah batch        : 15
  Ukuran per batch    : ~1226 file
  Istirahat per batch : 5 menit
  Total istirahat     : 70 menit


  ▶ Batch 1/15  (1226 file)
    Mulai : 21:17:05



  Batch 01:   0%|                          | 0/1226 [00:00<?, ?file/s]d:\doc\2026\FINAL CODE\skripsi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  Batch 01:  54%|████████▋       | 662/1226 [44:51<57:07,  6.08s/file]

  [SKIP] 091b7b0a80 — bad allocation (MemoryError)


  Batch 01:  54%|████████▋       | 665/1226 [45:06<51:22,  5.49s/file]

  [SKIP] 09286cc39a — bad allocation (MemoryError)


  Batch 01:  54%|████████▋       | 665/1226 [45:06<51:22,  5.49s/file]

  [SKIP] 092dcbbad6 — bad allocation (MemoryError)


  Batch 01:  54%|████████▋       | 666/1226 [45:09<44:55,  4.81s/file]